In [ ]:
"""
Heatwave Detection Core Functions

Detects heatwave events using the 3+ consecutive days ≥ 95th percentile threshold definition.
Caller handles data loading, spatial chunking, and I/O operations.
"""

import numpy as np
import pandas as pd
from typing import Tuple, Union


def detect_heatwave_events(
    values: np.ndarray,
    thresholds: np.ndarray,
    min_duration: int = 3
) -> np.ndarray:
    """
    Detect heatwave events from time series using threshold exceedance streaks.
    
    Identifies periods where values exceed thresholds for ≥ min_duration consecutive days.
    
    Parameters
    ----------
    values : np.ndarray
        1D array of daily temperature values
    thresholds : np.ndarray
        1D array of daily 95th percentile thresholds (same length as values)
    min_duration : int
        Minimum consecutive days above threshold to qualify as heatwave (default: 3)
    
    Returns
    -------
    np.ndarray
        Binary array (0/1) indicating heatwave occurrence at each timestep
    """
    # Handle NaNs: treat missing values as non-exceedances
    valid_mask = ~np.isnan(values) & ~np.isnan(thresholds)
    exceedance = np.zeros(len(values), dtype=bool)
    exceedance[valid_mask] = values[valid_mask] >= thresholds[valid_mask]
    
    # Run-length encoding to identify consecutive exceedance streaks
    streak_boundaries = np.insert(exceedance[:-1] != exceedance[1:], 0, False)
    streak_ids = np.cumsum(streak_boundaries)
    
    # Count streak lengths
    streak_lengths = np.bincount(streak_ids, weights=exceedance.astype(int))
    streak_durations = streak_lengths[streak_ids]
    
    # Flag days in qualifying streaks
    heatwave_mask = (streak_durations >= min_duration) & exceedance
    return heatwave_mask.astype(int)


def process_grid_cell_heatwave(
    temperature_ts: np.ndarray,
    p95_doy: np.ndarray,
    dates: pd.DatetimeIndex,
    min_duration: int = 3
) -> np.ndarray:
    """
    Detect heatwaves for a single grid cell using day-of-year percentile thresholds.
    
    Aligns daily 95th percentile thresholds to dates via day-of-year mapping before detection.
    
    Parameters
    ----------
    temperature_ts : np.ndarray
        1D array of daily maximum temperatures (time dimension only)
    p95_doy : np.ndarray
        366-element array of 95th percentile thresholds indexed by day-of-year
        (position 0 = Jan 1, position 365 = Dec 31)
    dates : pd.DatetimeIndex
        Dates corresponding to each temperature value
    min_duration : int
        Minimum consecutive days above threshold (default: 3)
    
    Returns
    -------
    np.ndarray
        Binary heatwave occurrence array aligned to input dates
    """
    # Map day-of-year to thresholds (handle leap years: Dec 31 = index 365 for both)
    day_of_year = dates.dayofyear.values - 1  # Convert to 0-based index
    day_of_year[day_of_year == 365] = 364      # Map leap day Dec 31 to non-leap index
    
    # Clip to valid range for non-leap years (365 days)
    day_of_year = np.clip(day_of_year, 0, 364)
    
    # Align thresholds to dates
    thresholds = p95_doy[day_of_year]
    
    # Detect events
    return detect_heatwave_events(
        values=temperature_ts,
        thresholds=thresholds,
        min_duration=min_duration
    )